In [1]:
import json
with open("1000_prompts.json", "r") as f:
    questions = json.load(f)

print(f"Loaded {len(questions)} prompts")

Loaded 1000 prompts


In [2]:
import pandas as pd
from vllm import LLM, SamplingParams
llm = LLM(
    model="mistralai/Mistral-7B-Instruct-v0.1", 
    tensor_parallel_size=1,
    gpu_memory_utilization=0.90,
    enforce_eager=True
)

sampling_params = SamplingParams(
    n=20,                
    temperature=0.8,
    top_p=0.95,
    max_tokens=200
)

INFO 03-25 13:02:07 [utils.py:233] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'model': 'mistralai/Mistral-7B-Instruct-v0.1'}
INFO 03-25 13:02:09 [model.py:533] Resolved architecture: MistralForCausalLM
INFO 03-25 13:02:09 [model.py:1582] Using max model len 32768
INFO 03-25 13:02:09 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-25 13:02:09 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 03-25 13:02:09 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-25 13:02:09 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-25 13:02:10 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 03-25 13:02:10 [compilation.py:289] Enabled custom fusions: norm_quant, act_quant
(EngineCore pid=2406) INFO 03-25 13:02

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=2406) INFO 03-25 13:02:21 [default_loader.py:384] Loading weights took 4.43 seconds
(EngineCore pid=2406) INFO 03-25 13:02:22 [gpu_model_runner.py:4566] Model loading took 13.5 GiB memory and 6.792924 seconds
(EngineCore pid=2406) INFO 03-25 13:02:26 [gpu_worker.py:456] Available KV cache memory: 28.24 GiB
(EngineCore pid=2406) INFO 03-25 13:02:26 [kv_cache_utils.py:1316] GPU KV cache size: 231,376 tokens
(EngineCore pid=2406) INFO 03-25 13:02:26 [kv_cache_utils.py:1321] Maximum concurrency for 32,768 tokens per request: 18.80x
(EngineCore pid=2406) INFO 03-25 13:02:26 [core.py:281] init engine (profile, create kv cache, warmup model) took 3.84 seconds
(EngineCore pid=2406) WARNING 03-25 13:02:28 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=2406) WARNING 03-25 13:02:28 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings tha

In [3]:
def generate_batch(prompts):
    outputs = llm.generate(prompts, sampling_params)

    results = []

    for i, out in enumerate(outputs):
        prompt = prompts[i]

        # 🔥 DEBUG: ensure 20 outputs
        print(f"Prompt {i} → {len(out.outputs)} outputs")

        for j, candidate in enumerate(out.outputs):
            results.append({
                "question": prompt,
                "candidate_id": j,
                "output": candidate.text.strip()
            })

    return results

def run_generation(questions, batch_size=64):
    all_results = []

    total = len(questions)

    for i in range(0, total, batch_size):
        batch = questions[i:i + batch_size]

        print(f"\n🚀 Processing {i} → {i + len(batch)}")

        batch_results = generate_batch(batch)
        all_results.extend(batch_results)

        # 🔥 Save checkpoint (VERY IMPORTANT)
        if i % 100 == 0 and i != 0:
            df_temp = pd.DataFrame(all_results)
            df_temp.to_json("checkpoint.jsonl", orient="records", lines=True)
            print("💾 Checkpoint saved")

    return all_results    

In [ ]:
results = run_generation(questions, batch_size=64)


🚀 Processing 0 → 64


Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Prompt 0 → 20 outputs
Prompt 1 → 20 outputs
Prompt 2 → 20 outputs
Prompt 3 → 20 outputs
Prompt 4 → 20 outputs
Prompt 5 → 20 outputs
Prompt 6 → 20 outputs
Prompt 7 → 20 outputs
Prompt 8 → 20 outputs
Prompt 9 → 20 outputs
Prompt 10 → 20 outputs
Prompt 11 → 20 outputs
Prompt 12 → 20 outputs
Prompt 13 → 20 outputs
Prompt 14 → 20 outputs
Prompt 15 → 20 outputs
Prompt 16 → 20 outputs
Prompt 17 → 20 outputs
Prompt 18 → 20 outputs
Prompt 19 → 20 outputs
Prompt 20 → 20 outputs
Prompt 21 → 20 outputs
Prompt 22 → 20 outputs
Prompt 23 → 20 outputs
Prompt 24 → 20 outputs
Prompt 25 → 20 outputs
Prompt 26 → 20 outputs
Prompt 27 → 20 outputs
Prompt 28 → 20 outputs
Prompt 29 → 20 outputs
Prompt 30 → 20 outputs
Prompt 31 → 20 outputs
Prompt 32 → 20 outputs
Prompt 33 → 20 outputs
Prompt 34 → 20 outputs
Prompt 35 → 20 outputs
Prompt 36 → 20 outputs
Prompt 37 → 20 outputs
Prompt 38 → 20 outputs
Prompt 39 → 20 outputs
Prompt 40 → 20 outputs
Prompt 41 → 20 outputs
Prompt 42 → 20 outputs
Prompt 43 → 20 output

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

...

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Prompt 0 → 20 outputs
Prompt 1 → 20 outputs
Prompt 2 → 20 outputs
Prompt 3 → 20 outputs
Prompt 4 → 20 outputs
Prompt 5 → 20 outputs
Prompt 6 → 20 outputs
Prompt 7 → 20 outputs
Prompt 8 → 20 outputs
Prompt 9 → 20 outputs
Prompt 10 → 20 outputs
Prompt 11 → 20 outputs
Prompt 12 → 20 outputs
Prompt 13 → 20 outputs
Prompt 14 → 20 outputs
Prompt 15 → 20 outputs
Prompt 16 → 20 outputs
Prompt 17 → 20 outputs
Prompt 18 → 20 outputs
Prompt 19 → 20 outputs
Prompt 20 → 20 outputs
Prompt 21 → 20 outputs
Prompt 22 → 20 outputs
Prompt 23 → 20 outputs
Prompt 24 → 20 outputs
Prompt 25 → 20 outputs
Prompt 26 → 20 outputs
Prompt 27 → 20 outputs
Prompt 28 → 20 outputs
Prompt 29 → 20 outputs
Prompt 30 → 20 outputs
Prompt 31 → 20 outputs
Prompt 32 → 20 outputs
Prompt 33 → 20 outputs
Prompt 34 → 20 outputs
Prompt 35 → 20 outputs
Prompt 36 → 20 outputs
Prompt 37 → 20 outputs
Prompt 38 → 20 outputs
Prompt 39 → 20 outputs
Prompt 40 → 20 outputs
Prompt 41 → 20 outputs
Prompt 42 → 20 outputs
Prompt 43 → 20 output

Rendering prompts:   0%|          | 0/64 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [6]:
df = pd.DataFrame(results)

# Save as JSONL (best for large data)
df.to_json("outputs_20_per_prompt.jsonl", orient="records", lines=True)

print("✅ Saved 20 outputs per prompt")

✅ Saved as JSON!
